## See Corner Structure with Harris

# Introduction to Corner Detection

In the first unit of this course, we outlined the image stitching pipeline and learned that to stitch images together, we must first find unique "landmarks" across our images.

Imagine trying to stitch together two pictures of a perfectly blank, blue sky. It would be nearly impossible because there are no unique reference points to tell you how the images align. However, if the images contain a highly textured brick building, the sharp edges and corners make alignment much easier.

In this second unit out of our five-unit journey, we will explore the **Harris Corner Detector**. The Harris algorithm is a "detector-only" method. It does not describe or match features across images; instead, it is a fantastic diagnostic tool. It shows us exactly where an algorithm finds corner-like structures. By building this tool, you will develop a strong intuition for why certain images are great for stitching, while others are prone to failure.

---

## Preparing the Image Data

Before we can look for corners, we need to load and prepare our image. As a quick reminder from our previous discussions, we will use our custom `cvkit` helper library to read our image file and prepare it.

```python
import cv2
import numpy as np
from cvkit import preprocess_for_features, read_color

image = read_color("sample_image.jpg")
gray = preprocess_for_features(image)

```

In this snippet, `read_color` safely loads our image. We then pass it to `preprocess_for_features`, which converts the image to grayscale. Corner detection relies on measuring drastic changes in light intensity (light to dark), so color information is not necessary and would only slow down our math.

Next, OpenCV's mathematical functions for finding corners require our grayscale image data to be in a specific format called a **32-bit float**. We can convert our grayscale image using NumPy:

```python
gray_float = np.float32(gray)

```

Now, our image is perfectly prepared for the Harris Corner Detector.

---

## Applying the Harris Corner Detector

To find corners, we will use the `cv2.cornerHarris()` function provided by OpenCV. This function scans the image to find areas where the pixel intensity changes significantly in all directions — the classic definition of a corner.

```python
response = cv2.cornerHarris(gray_float, blockSize=2, ksize=5, k=0.07)

```

Let us break down the parameters we just passed into the function:

* **`gray_float`**: This is our prepared, 32-bit float grayscale image.
* **`blockSize`**: This determines the size of the neighborhood the algorithm looks at. A value of `2` means it looks at a 2x2 pixel grid to detect corners.
* **`ksize`**: This is the aperture parameter used to mathematically find the edges. A value of `5` is a standard starting point.
* **`k`**: This is a mathematical parameter used to calculate the final "corner score." A standard value is `0.07`.

The `response` we get back is not an image we can display. Instead, it is a **map of scores**. Every pixel gets a score indicating how "corner-like" it is.

---

## Highlighting and Masking Corners

Because the highest-scoring points in our response map are often just a single pixel wide, they can be very hard to see. We can use OpenCV's `cv2.dilate()` function to slightly enlarge these bright spots.

```python
response = cv2.dilate(response, None)

```

Next, we want to isolate only the best, strongest corners. We can do this by setting a threshold. We will create a mask that only selects pixels where the score is greater than 1% (`0.01`) of the highest score found in the entire image.

```python
mask = response > 0.01 * response.max()

```

Now that we know exactly where our strongest corners are, let us highlight them on a copy of our original color image. We will color these pixels bright red so they stand out. Keep in mind that OpenCV uses BGR (Blue, Green, Red) color ordering, so red is represented as `[0, 0, 255]`.

```python
output = image.copy()
output[mask] = [0, 0, 255]

```

We can wrap all of this logic into a clean, reusable function that returns both the highlighted image and the total number of corners it found.

```python
def harris_overlay(image, gray):
    response = cv2.cornerHarris(np.float32(gray), blockSize=2, ksize=5, k=0.07)
    response = cv2.dilate(response, None)
    
    if response.max() <= 0:
        return image.copy(), 0
        
    mask = response > 0.01 * response.max()
    output = image.copy()
    output[mask] = [0, 0, 255]
    
    return output, int(mask.sum())

```

---

## Putting It All Together

Now, we can integrate our overlay function into a `main` script. This script loads the image, processes it, prints helpful diagnostic counts to the terminal, and displays the results side-by-side using our `cvkit.make_panel` helper function.

```python
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = read_color(args.path)
    gray = preprocess_for_features(image)
    corners, count = harris_overlay(image, gray)

    print("corner-like pixels:", count)
    print("note: this is a texture diagnostic, not the final matching method")

    cv2.imshow(
        "harris",
        make_panel(
            [
                ("feature input", gray),
                ("harris corner structure", corners),
            ],
            max_size=args.preview_size,
        ),
    )
    cv2.waitKey(0)
    cv2.destroyAllWindows()

```

If you run this code on an image of a textured building, you may see output in your terminal similar to this:

```text
corner-like pixels: 4521
note: this is a texture diagnostic, not the final matching method

```

A window will also pop up, showing the grayscale image on the left and the original image covered with red corner highlights on the right.

---

## Summary and Next Steps

In this lesson, you built a powerful diagnostic tool from scratch. You learned how to prepare an image, apply the Harris Corner Detector, enlarge the mathematical responses, and filter them using a threshold to create a visual overlay. By seeing where an image holds the most texture, you now have a foundational intuition for what makes an image suitable for panorama stitching.

Remember, the Harris algorithm is just a detector. It shows us the structure, but it cannot map points from one image to another.

In the interactive practice exercises coming up next, you will have the opportunity to write this code yourself. You will practice preparing the image arrays, tuning the `cv2.cornerHarris()` parameters, and applying the color masks to solidify these concepts before we move on to actual feature matching in the next unit.

## Setting Up the Command Line Interface

Now that the lesson has walked through the Harris corner pipeline, it is time to start building your own corner-finding script piece by piece.

Every good command-line tool starts by reading what the user wants, so your first job is to wire up argparse inside main(). The script will eventually take an image path and a preview size, so let's set those up now.

Inside main(), do the following:

    Create an ArgumentParser and store it in a variable called parser.
    Add a positional argument named path.
    Add an optional argument --preview-size with type=int and default=900.
    Call parser.parse_args() and save the result in args.

Once this foundation is in place, you'll be ready to plug in image loading and Harris detection in the next steps.

```
import argparse
import cv2
import numpy as np

from cvkit import make_panel, preprocess_for_features, read_color


def main():
    # TODO: Set up argument parsing for our script.
    # 1. Create an ArgumentParser instance and store it in a variable named "parser".
    # 2. Add a positional argument named "path".
    # 3. Add an optional argument "--preview-size" with type=int and default=900.
    # 4. Call parser.parse_args() and store the result in a variable named "args".
    pass


if __name__ == "__main__":
    main()

```

Here is the completed implementation for setting up the command-line interface with `argparse`:

```python
import argparse
import cv2
import numpy as np

from cvkit import make_panel, preprocess_for_features, read_color


def main():
    # 1. Create an ArgumentParser instance
    parser = argparse.ArgumentParser()

    # 2. Add a positional argument named "path"
    parser.add_argument("path")

    # 3. Add an optional argument "--preview-size" with type=int and default=900
    parser.add_argument("--preview-size", type=int, default=900)

    # 4. Call parser.parse_args() and store the result in "args"
    args = parser.parse_args()


if __name__ == "__main__":
    main()

```

## Painting Corners onto the Image

Nice work getting the CLI scaffolding in place. Now, it's time to bring the script to life by teaching it to actually find corners.

In this exercise, you'll fill in the body of the harris_overlay function so it returns a highlighted image and a count of corner pixels.

Here's what your function should do:

    Call cv2.cornerHarris on np.float32(gray) with blockSize=2, ksize=5, and k=0.07, storing the result in response.
    Dilate response with cv2.dilate(response, None) so the corner peaks become easier to see.
    If response.max() <= 0, return (image.copy(), 0) as a safety guard for flat images.
    Build a boolean mask of pixels where response is greater than 0.01 * response.max().
    Copy the input image and paint the masked pixels red. Remember OpenCV uses BGR, so red is [0, 0, 255].
    Return the highlighted image together with int(mask.sum()) as the corner count.

Take your time on the masking step — it's the heart of how Harris turns a response map into something you can actually see.

```
import argparse
import cv2
import numpy as np

from cvkit import make_panel, preprocess_for_features, read_color


def harris_overlay(image, gray):
    # TODO: Build the Harris corner overlay step by step.
    # 1. Run cv2.cornerHarris on np.float32(gray) with blockSize=2, ksize=5, k=0.07,
    #    and store the result in a variable named "response".
    # 2. Dilate "response" using cv2.dilate(response, None) and reassign it.
    # 3. Safety check: if response.max() <= 0, return (image.copy(), 0).
    # 4. Build a mask of pixels where response is greater than 1% of response.max().
    # 5. Make a copy of the input image and paint the masked pixels red.
    #    Remember OpenCV uses BGR ordering, so red is [0, 0, 255].
    # 6. Return the highlighted image and the corner count as an int (use mask.sum()).
    return image.copy(), 0


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()


if __name__ == "__main__":
    main()
```

Here is the completed implementation of the `harris_overlay` function in your script:

```python
import argparse
import cv2
import numpy as np

from cvkit import make_panel, preprocess_for_features, read_color


def harris_overlay(image, gray):
    # 1. Run cv2.cornerHarris on np.float32(gray)
    response = cv2.cornerHarris(np.float32(gray), blockSize=2, ksize=5, k=0.07)

    # 2. Dilate response to make corner peaks easier to see
    response = cv2.dilate(response, None)

    # 3. Safety check: return early if no positive response exists
    if response.max() <= 0:
        return image.copy(), 0

    # 4. Build a mask for strong corners (> 1% of the maximum response)
    mask = response > 0.01 * response.max()

    # 5. Make a copy of the original image and paint masked pixels red (BGR: [0, 0, 255])
    output = image.copy()
    output[mask] = [0, 0, 255]

    # 6. Return the highlighted image and total count of corner pixels
    return output, int(mask.sum())


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()


if __name__ == "__main__":
    main()

```

## Wiring the Harris Pipeline Together

With the CLI parsed and the Harris overlay function ready, it's time to connect the pieces so that the script becomes a real tool.

Your job in this exercise is to finish main() so that it runs the full pipeline from a file path on disk to a side-by-side preview on the screen.

Work through the TODO list step by step:

    Read the color image with read_color and turn it into grayscale with preprocess_for_features.
    Pass both to harris_overlay and unpack the highlighted image and the corner count.
    Print the "corner-like pixels:" line followed by the count, and then the diagnostic note about this not being the final matching method.
    Build a two-tile panel with make_panel (using ("feature input", gray) and ("harris corner structure", corners)), forwarding args.preview_size as max_size, and show it with cv2.imshow("harris", ...).
    Finish with cv2.waitKey(0) and cv2.destroyAllWindows() so that the window behaves nicely.

After running the script, check the terminal first: the corner count should be an integer, and the note should remind you that Harris is only a texture diagnostic. Then inspect the preview window to see whether red pixels land on real corner-like structure.

```
import argparse
import cv2
import numpy as np

from cvkit import make_panel, preprocess_for_features, read_color


def harris_overlay(image, gray):
    response = cv2.cornerHarris(np.float32(gray), blockSize=2, ksize=5, k=0.07)
    response = cv2.dilate(response, None)

    if response.max() <= 0:
        return image.copy(), 0

    mask = response > 0.01 * response.max()
    output = image.copy()
    output[mask] = [0, 0, 255]
    return output, int(mask.sum())


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    # TODO: Finish wiring up main() so the script actually runs end-to-end.
    # 1. Load the color image with read_color(args.path) into a variable named "image".
    # 2. Convert it to grayscale with preprocess_for_features(image) into "gray".
    # 3. Call harris_overlay(image, gray) and unpack the result into "corners" and "count".
    # 4. Print two lines:
    #      - "corner-like pixels:" followed by the count
    #      - "note: this is a texture diagnostic, not the final matching method"
    # 5. Use cv2.imshow("harris", ...) with a make_panel of two tiles:
    #      ("feature input", gray) and ("harris corner structure", corners),
    #      and pass max_size=args.preview_size to make_panel.
    # 6. Call cv2.waitKey(0) to keep the window open, then cv2.destroyAllWindows() to clean up.


if __name__ == "__main__":
    main()

```

Here is the fully completed pipeline in `main()` connecting image loading, Harris corner detection, terminal feedback, and the side-by-side preview panel:

```python
import argparse
import cv2
import numpy as np

from cvkit import make_panel, preprocess_for_features, read_color


def harris_overlay(image, gray):
    response = cv2.cornerHarris(np.float32(gray), blockSize=2, ksize=5, k=0.07)
    response = cv2.dilate(response, None)

    if response.max() <= 0:
        return image.copy(), 0

    mask = response > 0.01 * response.max()
    output = image.copy()
    output[mask] = [0, 0, 255]
    return output, int(mask.sum())


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    # 1. Load the color image
    image = read_color(args.path)

    # 2. Convert image to grayscale feature input
    gray = preprocess_for_features(image)

    # 3. Detect corners and retrieve count
    corners, count = harris_overlay(image, gray)

    # 4. Print terminal diagnostics
    print("corner-like pixels:", count)
    print("note: this is a texture diagnostic, not the final matching method")

    # 5. Build two-tile panel and display preview window
    panel = make_panel(
        [
            ("feature input", gray),
            ("harris corner structure", corners),
        ],
        max_size=args.preview_size,
    )
    cv2.imshow("harris", panel)

    # 6. Wait for key press and clean up windows
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

## Tuning the Harris Detector Parameters